In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-2: Preferences, reward models and RLHF

## Where this session fits

> **The limitation we start from: RLLM-1's verifier needs a program that can check a completion. For style, tone and helpfulness no such program exists, but people can still say which of two completions is better.**

- **The question:** how do comparisons become a reward, and what breaks when we push on it?
- **Position: session 2 of the RL for LLMs series (RLLM-1 to RLLM-14)**
    - RLLM-1 named the sources of a reward, and Intro session C1 glossed reward models in one slide
    - this session is the full story of the first source
    - the next session, RLLM-3, lets a model do the comparing
- Reward sources: **people compare (RLHF) (RLLM-2)** | an AI compares (RLAIF) (RLLM-3) | a program checks (RLVR) (RLLM-5)
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**By the end you can**

1. say what preference data is and why comparisons can be easier to collect than scores
2. state the Bradley-Terry model, say where it enters RLHF (and where it does not), and compute a probability and a loss from two rewards
3. say how a reward model is built and trained
4. read the three stages of RLHF, and say how one reward per completion reaches the per-token machinery of the Intro's Part B
5. explain reward over-optimization, and why a KL penalty limits it but does not cure it
    - a **KL penalty** is a charge for drifting from the starting model (defined on the leash slide)


## RLHF on one page

**Why this section?** Before any detail, see where each part of this session sits in the whole pipeline.

> **RLHF has three stages: SFT, a reward model fitted to comparisons, then RL against that reward model. We follow the pipeline in order: stage 2, then stage 3.**

<img src="images/rllm2_pipeline.png" width=85%>

- **RLHF** is reinforcement learning from human feedback
- **SFT** is supervised fine-tuning on **demonstrations**: completions written by people
- **reward model: learn a reward from comparisons (stage 2)**
    - Bradley-Terry enters only here: it is the assumption that lets comparisons train a reward model
- **RL stage: use the reward model in RL, see what goes wrong, and limit it (stage 3)**
    - the limit is a **KL penalty**, a charge for drifting from the starting model
    - it does not use Bradley-Terry; it uses the reward model that stage 2 produced


## Reward model: preference data

**Why this section?** Stage 2 starts from data. If no program can score a completion, people can still say which is better.

> **A preference example is a triple: a prompt, a preferred completion, and a non-preferred completion.**

- a **reward** is a number whose size means something
- a **preference** says "A is better than B", with no size
- a **preference example** is $(\x, \y^+, \y^-)$: prompt, preferred completion, non-preferred completion
- the completions usually come from the SFT model, sampled twice or more per prompt (the practice described for InstructGPT, OpenAI's RLHF-trained assistant)
    - similar in spirit to the Intro's **off-policy**: the comparisons come from the SFT model
    - not from the policy as RL changes it, so the reward model sees a distribution shift
- the data has two uses
    - imitate the preferred completion (SFT)
    - **learn a reward from the contrast**, and optimize it with RL (this session)
- when a program can check the answer, RLLM-1's verifier is better
- next: why compare two completions, and not score one?

**Why compare two completions, and not score one?**

> **Choosing the better of two is usually an easier judgment than giving a number (our reading).**

<img src="images/rllm2_compare.png" width=85%>

- raters may use a score scale differently, but tend to agree on which is better (our reading)
- both replies are acceptable: the data records an ordering, not correctness
- the price: a comparison carries less information than a number (our reading), and raters still disagree on some pairs
- another price, planted for RLLM-3: **human comparisons cost money and time**
    - every pair needs a paid rater to read two completions (our reading)
- next: how do comparisons become a reward?


## Reward model: the Bradley-Terry model

**Why this section?** We are training a reward model, but we never see rewards, only which completion won. We need a model of how a reward gap makes a win likely.

> **Bradley-Terry is an assumption that links unseen rewards to observed winners: the probability that $\y^+$ wins is the sigmoid of the reward gap. It is not a constraint added to the loss; the loss (next slide) is derived from it.**

- **the network scores each completion alone, and only the training compares two**
- the **reward model** $\rew_\psi(\x, \y)$ gives one number to a whole completion, with parameters $\psi$
    - $\psi$, not $\phi$, which Intro session B3 used for the critic
    - for now it is a black box
    - the section "Reward model: the network and its data" builds it
- the **Bradley-Terry model is an assumption**, not a fact about raters
    - each completion has one scalar quality, and a comparison is a noisy read of the gap
    - one pool of raters, and transitivity: if A beats B and B beats C, A beats C
    - real preferences are often intransitive and depend on the rater, so the model is an assumption
- the **Bradley-Terry model**: $P(\y^+ \succ \y^-) = \sigma\bigl(\rew_\psi(\x, \y^+) - \rew_\psi(\x, \y^-)\bigr)$
    - $\sigma$ is the sigmoid, and $\succ$ reads "is preferred to"
    - a sigmoid of a difference is 0.5 at a tie, rises toward 1 as the gap grows, and sees only the gap
- next: which $\psi$ makes the observed winners likely?

**Bradley-Terry by numbers**

- rewards $2.0$ for $\y^+$ and $0.5$ for $\y^-$, a gap of $1.5$
- $P(\y^+ \succ \y^-) = \sigma(1.5) \approx 0.82$
- in the reverse order, $\sigma(-1.5) \approx 0.18$
- a larger gap means a more confident preference: at a gap of $0$ the probability is $0.5$
- only the gap enters, so the reward is known **only up to a constant per prompt**
    - add the same number to both rewards and nothing changes


## Reward model: the loss

**Why this section?** The model has parameters $\psi$ and we have data. We need a rule for choosing $\psi$, and that rule is the loss.

> **The loss is the negative log-likelihood of the observed winners under Bradley-Terry. It is binary classification, with ordinary cross-entropy as the loss.**

- **setup:** a dataset of $N$ examples $(\x, \y^+, \y^-)$, and we always write the winner first
- the Bradley-Terry model gives the chance of each observed winner: $\sigma\bigl(\rew_\psi(\x, \y^+) - \rew_\psi(\x, \y^-)\bigr)$
- pick $\psi$ to make all observed winners likely: **maximum likelihood**
    - take the negative log of that chance, so a likely winner gives a small loss
    - average over the $N$ examples
- the **loss**, summing over the $N$ examples $i$ (the index is left implicit): $-\dfrac{1}{N} \sum_{i} \log \sigma\bigl(\rew_\psi(\x, \y^+) - \rew_\psi(\x, \y^-)\bigr)$

**The classifier view, by numbers**

- this is ordinary binary classification: "which of the two completions won?"
    - the reward difference is the **logit** (the raw score before the sigmoid turns it into a probability)
    - the loss is the usual binary cross-entropy, as in logistic regression
- with the example gap of $1.5$: the loss is $-\log 0.82 \approx 0.20$
- in the reverse order the loss is about $1.70$: a wrong ranking costs far more

**What the loss does**

<img src="images/rllm2_bradley_terry.png" width=75%>

- only **differences** matter: adding a constant to every reward of a prompt changes nothing
    - so the reward has no absolute zero ("RL stage: the leash" uses this: only gaps matter)
- right panel: the loss reaches zero only as the gap grows without limit
    - so on a pair the raters got wrong, training keeps pushing the gap, and the model can overfit noisy labels (our reading)
- next: what network produces the reward $\rew_\psi$?


## Reward model: the network and its data

**Why this section?** The reward model is itself a trained network, with choices that matter.

> **A reward model is usually a language model with a scalar output, trained on comparisons.**

- **architecture:** a language model (often the SFT model) whose next-token output is replaced by a **scalar head**, a final layer with one output number
    - read at the last token, which has seen the whole completion
- **data:** several completions per prompt, ranked by raters
    - $m$ ranked completions give $m(m-1)/2$ pairs (4 give 6)
    - those pairs are correlated, so more prompts help more than more pairs (our reading)
- **training:** the loss of the previous section on those pairs
- **what it does not do:** it does not know the truth, only what raters tended to prefer
    - on held-out human comparisons a learned reward model agrees well below 100% of the time
- weaknesses: a preference for long completions has been reported for learned reward models
    - it is reliable only near its training completions (our reading)
- stage 2 is done: we hold a reward model. Next, stage 3 uses it


## RL stage: one reward per completion, at the token level

**Why this section?** We hold a reward model and now use it in RL. the Intro's Part B credited every token, but the reward model scores a whole completion. How does one reward meet the tokens?

> **A completion is a sequence of tokens. The reward model scores only the whole completion, so the reward arrives at the last token.**

<img src="images/rllm2_mdp.png" width=78%>

- **state** = the prompt plus the tokens so far (the prefix)
- **action** = the next token
- two views of the same objective
    - a **bandit** over whole completions: one action, one reward
    - a **token-level process** with a delayed reward: zero until the last token
- same objective, different machinery: a critic, GAE and reward-to-go only make sense token by token
- RLLM-4 and RLLM-6 rely on the token view
- the KL penalty also lands token by token ("Where the penalty arrives")
- next: what goes wrong when the policy chases this reward?


## RL stage: the problem, reward over-optimization

**Why this section?** The reward model is a proxy for what people want. Pushing hard on it can make the real quality worse.

> **Pushed hard, the policy finds completions the reward model over-rates, and measured quality falls: reward over-optimization. That the same holds for people is our reading.**

<img src="images/rllm2_overoptimization.png" width=62%>

- schematic, invented curves
- the shape is the one [Gao et al. (2022, ICML 2023)](https://arxiv.org/abs/2210.10760) measured
    - a fixed **gold** reward model (a large model treated as the ground truth) stood in for people
    - proxy reward models were trained on its labels
- the reward the policy is trained on keeps rising, and the gold reward peaks, then falls
- why: the reward model learned from completions the SFT model wrote ("Reward model: preference data"), and the policy drifts away from them (our reading)

**Over-optimization and reward hacking**

- how a policy over-rates: it drifts to what the proxy likes and people do not
    - for example extra length or a confident tone (our reading)
- **reward over-optimization:** the fall above, from a learned reward model pushed too hard
- **reward hacking:** any exploit of a flawed reward
    - over-optimization is the case where the flaw is a learned proxy
    - RLLM-5 meets hacking of a rule-based reward
- the question for the next step: how do we keep the policy where the reward model can be trusted?


## RL stage: the leash, a reference model and a KL penalty

**Why this section?** The reward model is reliable only near the SFT model's completions. So stage 3 keeps the policy near its start.

> **Stage 3 adds a KL penalty for drift from a frozen reference model. A coefficient $\beta$ sets how short the leash is.**

- **the objective, in words:** maximize the reward model's reward, minus $\beta$ times the KL divergence from the reference model
- the **reference model** $\pi_{\text{ref}}$ is a frozen copy of the starting policy (here the SFT model)
- the **KL divergence** (Kullback-Leibler) measures how far two distributions are apart. In PPO the penalty at each token is
    - the token index: $\y_{\tt}$ is the token at position $\tt$ of the completion $\y$, and $\y_{<\tt}$ is the tokens before it (the prefix)
    - $-\beta \, d$ with $d = \log \dfrac{\pi_\theta(\y_{\tt} \mid \x, \y_{<\tt})}{\pi_{\text{ref}}(\y_{\tt} \mid \x, \y_{<\tt})}$ (Intro session C1)
    - sign convention: $d > 0$ where the policy has become more likely than the reference, so it is charged
    - for one sampled token it is a **sample estimate**, and the log-ratio can have either sign
    - its expected value over the sampled tokens is the KL, and summed over positions it is the sequence KL

**Reading the penalty: $\beta$ and the best policy**

- $\beta$ is the KL coefficient: large stays close, small chases the reward
    - only the size of the reward gaps relative to $\beta$ matters, not a constant shift of the reward
- a known result of this objective (our derivation not shown): the best policy is proportional to $\pi_{\text{ref}}(\y \mid \x) \, e^{\rew/\beta}$
    - this explains why only gaps relative to $\beta$ matter, and why $\beta$ has no formula
    - RLLM-10 (DPO) starts from this

**Where the penalty arrives, and what the leash costs**

<img src="images/rllm2_token_kl.png" width=60%>

- (schematic: invented sizes)

- the reward arrives once, at the last token ("RL stage: one reward per completion"), and the Intro's Part B machinery spreads the credit
    - PPO: the critic and GAE (Intro session B3, actor-critic) carry credit from the last token back
    - GRPO: the completion's advantage (Intro session C2, GRPO) applies to all its tokens
    - nothing new in the optimizer: the reward model is one more source of reward
- in PPO the KL penalty arrives at every token (in GRPO it is a loss term instead)
- stage 3 now holds **three models**: the policy, the reference and the reward model
    - PPO adds a critic (the old policy is a stored snapshot, not a network): four networks, counted fully in RLLM-4

**Why the leash limits but does not cure**

- it limits: a short leash keeps the policy where the reward model saw data
- it does not cure (our reading)
    - the reference is only an SFT model, not the truth: staying near it keeps its flaws
    - the reward model can still be wrong near the reference
    - $\beta$ has no formula: it is tuned by trial, and too large a $\beta$ stops learning
- the leash limits drift from the reference (KL)
    - Gao et al. found that at a given KL the gold reward did not measurably improve with the penalty, in their setup
    - it acted like stopping earlier (possibly hyperparameter-sensitive)


# What we established, and what it sets up

- **Established**
    - preference data records comparisons $(\x, \y^+, \y^-)$, often easier to collect than scores
    - Bradley-Terry: the probability of a preference is the sigmoid of a reward difference, and fitting it is ordinary cross-entropy
    - it is used only to fit the reward model (stage 2)
    - a reward model is a language model with a scalar head, trained on comparisons
    - RLHF: SFT, a reward model, then RL, and the reward arrives at the last token and feeds the Intro's Part B per-token machinery
    - pushing hard on a reward model over-optimizes it, and a KL leash to a reference model limits this
- **Sets up**
    - **the limit of RLHF:** a reward model is a proxy, and the leash does not cure that
    - human comparisons also cost money and time ("Reward model: preference data")
    - next session, RLLM-3: let a model do the comparing (RLAIF, reinforcement learning from AI feedback), guided by written principles
    - RLLM-4 opens the RL stage, and RLLM-10 teaches DPO (direct preference optimization), which learns from the same data without a separate reward model


## An example preference dataset (aside)

> **UltraFeedback (binarized) is a dataset of (prompt, chosen, rejected) triples, labeled by a model rather than by people.**

- [UltraFeedback (binarized)](https://huggingface.co/datasets/HuggingFaceH4/ultrafeedback_binarized): columns *prompt*, *chosen*, *rejected*
- built from prompts, with completions from several different language models
    - a strong language model (GPT-4) rates each completion, and the ratings rank them
    - **binarized:** the best-rated completion is *chosen*
    - one of the others, at random, is *rejected*
- a model, not people, supplies the comparisons: RLLM-3 studies that choice

## Origins of preference-based RL (aside)

- [Christiano et al. 2017](https://arxiv.org/abs/1706.03741): RL from human preferences
- [Ziegler et al. 2019](https://arxiv.org/abs/1909.08593) and [Stiennon et al. 2020](https://arxiv.org/abs/2009.01325): language models
- [Ouyang et al. 2022](https://arxiv.org/abs/2203.02155): InstructGPT
